In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# Find project root
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"

crsp_path = PROCESSED / "clean_crsp.parquet"
trans_path = PROCESSED / "transcript_quarter_panel.parquet"

print("Project root:", ROOT)
print("Processed folder:", PROCESSED)
print("CRSP exists?", crsp_path.exists())
print("Transcript exists?", trans_path.exists())

Project root: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project
Processed folder: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed
CRSP exists? True
Transcript exists? True


In [4]:
crsp = pd.read_parquet(crsp_path)
trans = pd.read_parquet(trans_path)

print("CRSP shape:", crsp.shape)
print("Transcript shape:", trans.shape)

print("\nCRSP columns:")
print(crsp.columns.tolist())

print("\nTranscript columns:")
print(trans.columns.tolist())

CRSP shape: (493203, 21)
Transcript shape: (8120, 7)

CRSP columns:
['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date']

Transcript columns:
['ticker', 'quarter', 'Numeric Transeprency ', 'analyst_selectivity_ratio', 'language_complexity', 'net_positivity', 'quarter_end']


In [5]:
crsp["ticker"] = crsp["ticker"].astype(str).str.upper().str.strip()
trans["ticker"] = trans["ticker"].astype(str).str.upper().str.strip()

In [6]:
crsp_tickers = set(crsp["ticker"].dropna().unique())
trans_tickers = set(trans["ticker"].dropna().unique())

common_tickers = sorted(crsp_tickers & trans_tickers)
only_crsp = sorted(crsp_tickers - trans_tickers)
only_trans = sorted(trans_tickers - crsp_tickers)

print("CRSP tickers:", len(crsp_tickers))
print("Transcript tickers:", len(trans_tickers))
print("Common tickers:", len(common_tickers))
print("Only in CRSP:", len(only_crsp))
print("Only in Transcript:", len(only_trans))

print("\nExamples only in CRSP:", only_crsp[:10])
print("Examples only in Transcript:", only_trans[:10])

CRSP tickers: 185
Transcript tickers: 187
Common tickers: 180
Only in CRSP: 5
Only in Transcript: 7

Examples only in CRSP: ['COR', 'CVX', 'EXPD', 'GOOG', 'SCHW']
Examples only in Transcript: ['ACN', 'AON', 'BF.B', 'EQR', 'ETN', 'IVZ', 'SLB']


In [7]:
crsp = crsp[crsp["ticker"].isin(common_tickers)].copy()
trans = trans[trans["ticker"].isin(common_tickers)].copy()

print("Filtered CRSP shape:", crsp.shape)
print("Filtered Transcript shape:", trans.shape)

Filtered CRSP shape: (479783, 21)
Filtered Transcript shape: (7812, 7)


In [8]:
crsp = crsp.sort_values(["ticker", "date"]).reset_index(drop=True)
trans = trans.sort_values(["ticker", "quarter_end"]).reset_index(drop=True)

print(crsp[["ticker", "date"]].head())
print(trans[["ticker", "quarter", "quarter_end"]].head())

  ticker       date
0      A 2014-01-02
1      A 2014-01-03
2      A 2014-01-06
3      A 2014-01-07
4      A 2014-01-08
  ticker  quarter quarter_end
0      A  CQ12014  2014-03-31
1      A  CQ22014  2014-06-30
2      A  CQ32014  2014-09-30
3      A  CQ42014  2014-12-31
4      A  CQ12015  2015-03-31


In [11]:
#Merge ticker by ticker (safer than one global merge_asof)

# Make sure date columns are proper datetime
crsp["date"] = pd.to_datetime(crsp["date"], errors="coerce")
trans["quarter_end"] = pd.to_datetime(trans["quarter_end"], errors="coerce")

# Drop bad rows
crsp = crsp.dropna(subset=["ticker", "date"]).copy()
trans = trans.dropna(subset=["ticker", "quarter_end"]).copy()

merged_parts = []

for ticker in common_tickers:
    crsp_t = crsp[crsp["ticker"] == ticker].copy()
    trans_t = trans[trans["ticker"] == ticker].copy()

    # sort within each ticker
    crsp_t = crsp_t.sort_values("date").reset_index(drop=True)
    trans_t = trans_t.sort_values("quarter_end").reset_index(drop=True)

    # skip if either side is empty
    if crsp_t.empty or trans_t.empty:
        continue

    merged_t = pd.merge_asof(
        crsp_t,
        trans_t,
        left_on="date",
        right_on="quarter_end",
        direction="backward",
        allow_exact_matches=True
    )

    merged_parts.append(merged_t)

# combine all tickers
master_panel = pd.concat(merged_parts, ignore_index=True)

# clean duplicate ticker columns created by merge
if "ticker_x" in master_panel.columns:
    master_panel = master_panel.rename(columns={"ticker_x": "ticker"})
if "ticker_y" in master_panel.columns:
    master_panel = master_panel.drop(columns=["ticker_y"])

# final sort
master_panel = master_panel.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Master panel shape:", master_panel.shape)
master_panel.head()

Master panel shape: (479783, 27)


,permno,permco,ticker,cusip,issuernm,siccd,naics,dlyclose,dlyopen,dlyhigh,...,sprtrn,vwretd,ewretd,date,quarter,Numeric Transeprency,analyst_selectivity_ratio,language_complexity,net_positivity,quarter_end
0,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.21,57.10,57.100,...,-0.008862,-0.008757,-0.004051,2014-01-02,NaN,NaN,NaN,NaN,NaN,NaT
1,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.92,56.39,57.345,...,-0.000333,0.000491,0.004096,2014-01-03,NaN,NaN,NaN,NaN,NaN,NaT
2,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.64,57.40,57.700,...,-0.002512,-0.003340,-0.001676,2014-01-06,NaN,NaN,NaN,NaN,NaN,NaT
3,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,57.45,56.95,57.630,...,0.006082,0.006090,0.006892,2014-01-07,NaN,NaN,NaN,NaN,NaN,NaT
4,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,58.39,57.33,58.540,...,-0.000212,0.000155,0.000835,2014-01-08,NaN,NaN,NaN,NaN,NaN,NaT


In [12]:
print(master_panel.columns.tolist())
print(master_panel.duplicated(subset=["ticker", "date"]).sum())

['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date', 'quarter', 'Numeric Transeprency ', 'analyst_selectivity_ratio', 'language_complexity', 'net_positivity', 'quarter_end']
0


In [13]:
print("Duplicate ticker-date rows:", master_panel.duplicated(subset=["ticker", "date"]).sum())

check_cols = [
    "ticker", "date", "quarter", "quarter_end",
    "net_positivity", "language_complexity"
]

available_check_cols = [c for c in check_cols if c in master_panel.columns]
master_panel[available_check_cols].head(10)

Duplicate ticker-date rows: 0


,ticker,date,quarter,quarter_end,net_positivity,language_complexity
0,A,2014-01-02,NaN,NaT,NaN,NaN
1,A,2014-01-03,NaN,NaT,NaN,NaN
2,A,2014-01-06,NaN,NaT,NaN,NaN
3,A,2014-01-07,NaN,NaT,NaN,NaN
4,A,2014-01-08,NaN,NaT,NaN,NaN
5,A,2014-01-09,NaN,NaT,NaN,NaN
6,A,2014-01-10,NaN,NaT,NaN,NaN
7,A,2014-01-13,NaN,NaT,NaN,NaN
8,A,2014-01-14,NaN,NaT,NaN,NaN
9,A,2014-01-15,NaN,NaT,NaN,NaN


In [14]:
sample_ticker = "AAPL"

sample = master_panel[master_panel["ticker"] == sample_ticker][
    ["ticker", "date", "quarter", "quarter_end",
     "net_positivity", "language_complexity"]
].tail(20)

sample

,ticker,date,quarter,quarter_end,net_positivity,language_complexity
5516,AAPL,2024-12-03,CQ32024,2024-09-30,1.57,12.04
5517,AAPL,2024-12-04,CQ32024,2024-09-30,1.57,12.04
5518,AAPL,2024-12-05,CQ32024,2024-09-30,1.57,12.04
5519,AAPL,2024-12-06,CQ32024,2024-09-30,1.57,12.04
5520,AAPL,2024-12-09,CQ32024,2024-09-30,1.57,12.04
5521,AAPL,2024-12-10,CQ32024,2024-09-30,1.57,12.04
5522,AAPL,2024-12-11,CQ32024,2024-09-30,1.57,12.04
5523,AAPL,2024-12-12,CQ32024,2024-09-30,1.57,12.04
5524,AAPL,2024-12-13,CQ32024,2024-09-30,1.57,12.04
5525,AAPL,2024-12-16,CQ32024,2024-09-30,1.57,12.04


In [15]:
output_path = PROCESSED / "master_panel.parquet"
master_panel.to_parquet(output_path, index=False)

print("Master panel saved to:", output_path)

print("Saved master panel to:")
print(output_path)

Master panel saved to: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\master_panel.parquet
Saved master panel to:
c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\master_panel.parquet


In [16]:
master_panel = master_panel.rename(columns={
    "Numeric Transeprency ": "numeric_transparency",
    "Numeric Transeprency": "numeric_transparency"
})

master_panel = master_panel.sort_values(["ticker", "date"]).reset_index(drop=True)

output_path = PROCESSED / "master_panel.parquet"
master_panel.to_parquet(output_path, index=False)

print("Saved cleaned master panel to:", output_path)
print(master_panel.columns.tolist())

Saved cleaned master panel to: c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\master_panel.parquet
['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date', 'quarter', 'numeric_transparency', 'analyst_selectivity_ratio', 'language_complexity', 'net_positivity', 'quarter_end']
